# Healthcare Diabetes-Risk API — Google Colab runner

Runs the **FastAPI + ML** demo inside Colab. Three Colab-specific things are handled for you:

1. **Multi-file package** → upload the project zip and unzip it.
2. **sklearn pickle compatibility** → retrain in Colab so the artifact matches Colab's sklearn.
3. **No auto-exposed ports** → verify in-process with `TestClient` (Step 3), or open a real `/docs` (Step 4).


## Step 1 — Upload the project zip
Run the cell, then choose **`healthcare_api_demo.zip`** from your computer.


In [ ]:
from google.colab import files
up = files.upload()   # select healthcare_api_demo.zip


In [ ]:
!unzip -oq healthcare_api_demo.zip
%cd healthcare_api_demo
!ls -R . | head -30


## Step 2 — Install deps and (re)train
Colab already has pandas/sklearn/joblib. We add FastAPI's stack, then retrain so the saved model matches Colab's sklearn version (the bundled data needs no internet).


In [ ]:
!pip install -q fastapi "uvicorn[standard]" httpx nest_asyncio


In [ ]:
!python train_model.py


## Step 3 — Verify the API in-process (no server needed)
`TestClient` exercises the real app — routing, validation, model — without opening a port. Note the **`with`** block: it triggers startup so the model loads.


In [ ]:
from fastapi.testclient import TestClient
from app.main import app
import json

with TestClient(app) as client:
    print('HEALTH    :', client.get('/health').json())

    info = client.get('/model-info').json()
    print('ALGORITHM :', info['algorithm'], '| AUC:', info['test_metrics']['roc_auc'])

    high = {'Pregnancies':6,'Glucose':148,'BloodPressure':72,'SkinThickness':35,
            'Insulin':0,'BMI':33.6,'DiabetesPedigreeFunction':0.627,'Age':50}
    low  = {'Pregnancies':1,'Glucose':85,'BloodPressure':66,'SkinThickness':29,
            'Insulin':0,'BMI':26.6,'DiabetesPedigreeFunction':0.351,'Age':31}
    print('PREDICT hi:', client.post('/predict', json=high).json())
    print('PREDICT lo:', client.post('/predict', json=low).json())

    bad = {**high, 'Glucose':999, 'Age':200}   # out of range
    r = client.post('/predict', json=bad)
    print('VALIDATION:', r.status_code, [e['loc'][-1] for e in r.json()['details']])


## Step 4 (optional) — Run a real server and open `/docs`
Starts uvicorn in a background thread, then opens the live Swagger UI in a Colab window. `nest_asyncio` is required because Colab already runs an event loop.


In [ ]:
import nest_asyncio, asyncio, threading, time, uvicorn
nest_asyncio.apply()
from app.main import app

config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning')
server = uvicorn.Server(config)
threading.Thread(target=lambda: asyncio.run(server.serve()), daemon=True).start()
time.sleep(3)
print('uvicorn listening on :8000')

from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8000, path='/docs')


In [ ]:
# Call the live server from the notebook too:
import requests
print(requests.get('http://localhost:8000/health').json())
print(requests.post('http://localhost:8000/predict', json={
    'Pregnancies':8,'Glucose':183,'BloodPressure':64,'SkinThickness':0,
    'Insulin':0,'BMI':23.3,'DiabetesPedigreeFunction':0.672,'Age':32}).json())


## Notes & troubleshooting
- **Want a public URL to share?** Use a tunnel instead of Step 4:
  `!pip install pyngrok` → `from pyngrok import ngrok; print(ngrok.connect(8000))` (needs a free ngrok authtoken), or download `cloudflared` and run `cloudflared tunnel --url http://localhost:8000` (no signup).
- **`signal only works in main thread`** → use the `uvicorn.Server` pattern in Step 4 (modern uvicorn skips signal handlers off the main thread); don't call `uvicorn.run()` directly.
- **Restarting Step 4** → `Runtime ▸ Restart session`, since the old thread keeps port 8000.
- **Skip the upload** → push the folder to GitHub and replace Step 1 with `!git clone <your-repo>` then `%cd`.
